<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>Análisis de Sentimiento con Redes Neuronales Profundas</h2>
</div>

Anteriormente implementamos la Regresión Logística y Naive Bayes para análisis de sentimientos. Sin embargo, si le dieramos a esos modelos anteriores un ejemplo como:

<center> <span style='color:blue'> <b>This movie was almost good.</b> </span> </center>

El modelo habría predicho un sentimiento positivo para esa reseña. Sin embargo, esa oración tiene un sentimiento negativo e indica que la película no fue buena. Para resolver ese tipo de clasificaciones erróneas, implementaremos un programa que usa redes neuronales profundas para identificar el sentimiento en el texto. 

Utilizaremos la librería de trax para esta implementación:

- El código fuente de Trax se puede encontrar en Github: [Trax](https://github.com/google/trax)
- El código de Trax también usa la librería JAX: [JAX](https://jax.readthedocs.io/en/latest/index.html)

# Importar librerías y probar Trax

- Importemos las librerías y veamos un ejemplo de uso de la librería Trax.

In [1]:
import os 
import random as rnd
import numpy as npy
# importar librerías relevantes
import trax

# establecer semillas aleatorias para hacer este notebook más fácil de replicar
rnd.seed(31)
npy.random.seed(31)
# importar trax.fastmath.numpy
import trax.fastmath.numpy as np

# importar trax.layers
from trax import layers as tl

# importar Layer del archivo utils.py
from utils import Layer, load_tweets, process_tweet
#from utils import 


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\uie70742\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\uie70742\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Crear un arreglo usando trax.fastmath.numpy
a = np.array(5.0)

# Ver el arreglo devuelto
display(a)

print(type(a))

Array(5., dtype=float32, weak_type=True)

<class 'jaxlib._jax.ArrayImpl'>


Observamos que trax.fastmath.numpy devuelve un DeviceArray de la librería jax.

In [3]:
# Definir una función que usará el arreglo de trax.fastmath.numpy
def f(x):
    # f = x^2
    return (x**2)

In [4]:
# Llamar a la función
print(f"f(a) for a={a} is {f(a)}")

f(a) for a=5.0 is 25.0


El gradiente (derivada) de la función `f` con respecto a su entrada `x` es la derivada de $x^2$.
- La derivada de $x^2$ es $2x$.  
- Cuando x es 5, entonces $2x=10$.

Podemos calcular el gradiente de una función usando `trax.fastmath.grad(fun=)` y pasando el nombre de la función.
- En este caso la función de la que queremos tomar el gradiente es `f`.
- El objeto devuelto (guardado en `grad_f` en este ejemplo) es una función que puede calcular el gradiente de f para un arreglo trax.fastmath.numpy dado.

In [5]:
# Usar directamente trax.fastmath.grad para calcular el gradiente (derivada) de la función
grad_f = trax.fastmath.grad(fun=f)  # df / dx - Gradiente de la función f(x) con respecto a x

# Ver el tipo del objeto devuelto (es una función)
type(grad_f)

function

In [6]:
# Llamar a la función recién creada y pasar un valor para x (el DeviceArray almacenado en 'a')
grad_calculation = grad_f(a)

# Ver el resultado de llamar a la función grad_f
display(grad_calculation)

Array(10., dtype=float32, weak_type=True)

La función devuelta por trax.fastmath.grad recibe x=5 y calcula el gradiente de f, que es 2*x, es decir 10. El valor también se almacena como un DeviceArray de la librería jax.

# Importar los datos para Análisis de Sentimientos

## Cargar los datos

Importamos el conjunto de datos.  


In [8]:
import numpy as np

# Cargar tweets positivos y negativos
all_positive_tweets, all_negative_tweets = load_tweets()

# Ver el número total de tweets positivos y negativos.
print(f"Número de tweets Positivos: {len(all_positive_tweets)}")
print(f"Número de tweets Negativos: {len(all_negative_tweets)}")

# Dividir el conjunto positivo en validación y entrenamiento
val_pos   = all_positive_tweets[4000:] # generando conjunto de validación para tweets positivos
train_pos  = all_positive_tweets[:4000]# generando conjunto de entrenamiento para tweets positivos

# Dividir el conjunto negativo en validación y entrenamiento
val_neg   = all_negative_tweets[4000:] # generando conjunto de validación para tweets negativos
train_neg  = all_negative_tweets[:4000] # generando conjunto de entrenamiento para tweets negativos

# Combinar los datos de entrenamiento en un solo conjunto
train_x = train_pos + train_neg 

# Combinar los datos de validación en un solo conjunto
val_x  = val_pos + val_neg

# Establecer las etiquetas para el conjunto de entrenamiento (1 para positivo, 0 para negativo)
train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))

# Establecer las etiquetas para el conjunto de validación (1 para positivo, 0 para negativo)
val_y  = np.append(np.ones(len(val_pos)), np.zeros(len(val_neg)))

print(f"Tamaño de train_x {len(train_x)}")
print(f"Tamaño de val_x {len(val_x)}")

Número de tweets Positivos: 5000
Número de tweets Negativos: 5000
Tamaño de train_x 8000
Tamaño de val_x 2000


Ahora utilizaremos una función que procesa tweets (esta función está en el archivo utils.py).
- `process_tweets` elimina caracteres no deseados, p. ej. hashtag, hipervínculos, tickers de acciones del tweet.
- También devuelve una lista de palabras (tokeniza la cadena original).

In [9]:
# Importar una función que procesa los tweets
# from utils import process_tweet

# Probar la función que procesa tweets
print("original tweet at training position 0")
print(train_pos[0])

print("Tweet at training position 0 after processing:")
process_tweet(train_pos[0])

original tweet at training position 0
#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
Tweet at training position 0 after processing:


['followfriday', 'top', 'engag', 'member', 'commun', 'week', ':)']

Observemos que la función `process_tweet` conserva las palabras clave, elimina el símbolo de almohadilla #, e ignora los nombres de usuario (palabras que comienzan con '@'). También devuelve una lista de las palabras.


## Construir el vocabulario

Ahora construiremos el vocabulario.
- Mapeando cada palabra de cada tweet a un entero (un "índice"). 
- Asignaremos un índice a cada palabra iterando sobre tu conjunto de entrenamiento.

El vocabulario también incluirá algunos tokens especiales
- `__PAD__`: relleno (padding)
- `</e>`: fin de línea
- `__UNK__`: un token que representa cualquier palabra que no esté en el vocabulario.

In [10]:
# Construir el vocabulario
# Incluir tokens especiales 
# comenzando con los tokens pad, fin de línea y unk
Vocab = {'__PAD__': 0, '__</e>__': 1, '__UNK__': 2} 

# Nota: construimos el vocabulario usando datos de entrenamiento


Total words in vocab are 9070


{'__PAD__': 0,
 '__</e>__': 1,
 '__UNK__': 2,
 'followfriday': 3,
 'top': 4,
 'engag': 5,
 'member': 6,
 'commun': 7,
 'week': 8,
 ':)': 9,
 'hey': 10,
 'jame': 11,
 'odd': 12,
 ':/': 13,
 'pleas': 14,
 'call': 15,
 'contact': 16,
 'centr': 17,
 '02392441234': 18,
 'abl': 19,
 'assist': 20,
 'mani': 21,
 'thank': 22,
 'listen': 23,
 'last': 24,
 'night': 25,
 'bleed': 26,
 'amaz': 27,
 'track': 28,
 'scotland': 29,
 'congrat': 30,
 'yeaaah': 31,
 'yipppi': 32,
 'accnt': 33,
 'verifi': 34,
 'rqst': 35,
 'succeed': 36,
 'got': 37,
 'blue': 38,
 'tick': 39,
 'mark': 40,
 'fb': 41,
 'profil': 42,
 '15': 43,
 'day': 44,
 'one': 45,
 'irresist': 46,
 'flipkartfashionfriday': 47,
 'like': 48,
 'keep': 49,
 'love': 50,
 'custom': 51,
 'wait': 52,
 'long': 53,
 'hope': 54,
 'enjoy': 55,
 'happi': 56,
 'friday': 57,
 'lwwf': 58,
 'second': 59,
 'thought': 60,
 '’': 61,
 'enough': 62,
 'time': 63,
 'dd': 64,
 'new': 65,
 'short': 66,
 'enter': 67,
 'system': 68,
 'sheep': 69,
 'must': 70,
 'buy':

- Cada palabra única tiene un entero único asociado a ella.
- El número total de palabras en Vocab: 9088

## Convertir un tweet a un tensor

La siguiente función convertirá cada tweet a un tensor (una lista de IDs enteros únicos que representan el tweet procesado).
- Nota, el tipo de dato devuelto será una **`list()` regular de Python**
    - No usaremos TensorFlow en esta función
    - Tampoco usaremos un arreglo de numpy
    - Tampoco usaremos un arreglo de trax.fastmath.numpy
- Para las palabras del tweet que no estén en el vocabulario, asígnalas al ID único del token `__UNK__`.

##### Ejemplo
Entrada de un tweet:
```CPP
'@happypuppy, is Maria happy?'
```

tweet_to_tensor primero convertiremos el tweet en una lista de tokens (incluyendo solo palabras relevantes)
```CPP
['maria', 'happi']
```

Luego convertiremos cada palabra en su entero único

```CPP
[2, 56]
```
- Observemos que la palabra "maria" no está en el vocabulario, así que se le asigna el entero único asociado al token `__UNK__`, porque se considera "desconocida".

La función `tweet_to_tensor` reciba un tweet y lo convierte en un arreglo de números. Usamos el diccionario `Vocab` para ayudar a crear el tensor. 


In [11]:
def tweet_to_tensor(tweet, vocab_dict, unk_token='__UNK__', verbose=False):
    # Procesar el tweet en una lista de palabras
    # donde solo se conservan las palabras importantes (stop words eliminadas)
    word_l = process_tweet(tweet)
    
    if verbose:
        print("Lista de palabras procesadas del tweet:")
        print(word_l)
        
    # Inicializar la lista que contendrá los IDs enteros únicos de cada palabra
    tensor_l = []
    
    # Obtener el ID entero único del token __UNK__
    unk_ID = vocab_dict[unk_token]
    
    if verbose:
        print(f"El único ID entero de unk_token es {unk_ID}")
        
    # para cada palabra en la lista:
    for word in word_l:
        # Obtener el ID entero único.
        # Si la palabra no existe en el diccionario de vocabulario,
        # usar el ID único de __UNK__ en su lugar.
        word_ID = vocab_dict[word] if word in vocab_dict else unk_ID
        # Agregar el ID entero único a la lista del tensor.
        tensor_l.append(word_ID) 
    
    return tensor_l

In [ ]:
# Probar un tweet a tensor



In [14]:
# probar tweet_to_tensor
def test_tweet_to_tensor():
    test_cases = [
        {
            "name":"simple_test_check",
            "input": [val_pos[1], Vocab],
            "expected":[444, 2, 304, 567, 56, 9],
            "error":"La función devuelve un resultado incorrecto para val_pos[1]. La prueba falló."
        },
        {
            "name":"datatype_check",
            "input":[val_pos[1], Vocab],
            "expected":type([]),
            "error":"Incompatibilidad de tipos de datos. Se requiere una lista, no un np.array."
        },
        {
            "name":"without_unk_check",
            "input":[val_pos[1], Vocab],
            "expected":6,
            "error":"No se ha realizado la comprobación de palabras desconocidas; por favor, verifique si ha incluido la asignación para la palabra desconocida."
        }
    ]
    count = 0
    for test_case in test_cases:
        
        try:
            if test_case['name'] == "simple_test_check":
                assert test_case["expected"] == tweet_to_tensor(*test_case['input'])
                count += 1
            if test_case['name'] == "datatype_check":
                assert isinstance(tweet_to_tensor(*test_case['input']), test_case["expected"])
                count += 1
            if test_case['name'] == "without_unk_check":
                assert None not in tweet_to_tensor(*test_case['input'])
                count += 1
                
            
            
        except:
            print(test_case['error'])
    if count == 3:
        print("\033[92m Todas las pruebas se superaron con éxito.")
    else:
        print(count," Pruebas aprobadas de 3")
test_tweet_to_tensor()            

La función devuelve un resultado incorrecto para val_pos[1]. La prueba falló.
2  Pruebas aprobadas de 3


## Crear un generador de lotes (batch generator)

La mayor parte del tiempo en el Procesamiento de Lenguaje Natural, y en la IA en general, usamos lotes (batches) al entrenar nuestros conjuntos de datos. 
- Si en lugar de entrenar con lotes de ejemplos, entrenamos un modelo con un ejemplo a la vez, tomaría muchísimo tiempo entrenar el modelo. 
- Construiremos un generador de datos que recibe los tweets positivos/negativos y devuelve un lote de ejemplos de entrenamiento. Devuelve las entradas del modelo, los objetivos (etiquetas positivas o negativas) y el peso de cada objetivo (ej.: esto nos permite tratar algunos ejemplos como más importantes de acertar que otros, pero comúnmente todos serán 1.0). 

Una vez que creemos el generador, podemos incluirlo en un bucle for

```CPP
for batch_inputs, batch_targets, batch_example_weights in data_generator:
    ...
```

También podemos obtener un solo lote así:

```CPP
batch_inputs, batch_targets, batch_example_weights = next(data_generator)
```


In [15]:
def data_generator(data_pos, data_neg, batch_size, loop, vocab_dict, shuffle=False):
   
    # aseguramos de que el tamaño del lote sea un número par
    # para permitir un número igual de muestras positivas y negativas
    assert batch_size % 2 == 0
    
    # El número de ejemplos positivos en cada lote es la mitad del tamaño del lote
    # lo mismo con el número de ejemplos negativos en cada lote
    n_to_take = batch_size // 2
    
    # Usar pos_index para recorrer el arreglo data_pos
    # lo mismo con neg_index y data_neg
    pos_index = 0
    neg_index = 0
    
    len_data_pos = len(data_pos)
    len_data_neg = len(data_neg)
    
    # Obtener un arreglo con los índices de los datos
    pos_index_lines = list(range(len_data_pos))
    neg_index_lines = list(range(len_data_neg))
    
    # mezclar las líneas si shuffle es True
    if shuffle:
        rnd.shuffle(pos_index_lines)
        rnd.shuffle(neg_index_lines)
        
    stop = False
    
    # Bucle indefinido
    while not stop:  
        # crear un lote con ejemplos positivos y negativos
        batch = []
        # Primera parte: Empaquetar n_to_take ejemplos positivos
        # Comenzar desde pos_index e incrementar i hasta n_to_take
        for i in range(n_to_take):
            # Si el índice positivo sobrepasa la longitud del conjunto de datos positivo,
            if pos_index >= len_data_pos: 
                # Si loop es False, romper una vez que llegamos al final del conjunto de datos
                if not loop:
                    stop = True;
                    break;
                # Si el usuario quiere seguir reutilizando los datos, reiniciar el índice
                pos_index = 0
                if shuffle:
                    # Mezclar el índice de la muestra positiva
                    rnd.shuffle(pos_index_lines)
            # obtener el tweet en pos_index
            tweet = data_pos[pos_index_lines[pos_index]]
            
            # convertir el tweet en tensores de enteros que representan las palabras procesadas
            tensor = tweet_to_tensor(tweet, vocab_dict)
            # agregar el tensor a la lista del lote
            batch.append(tensor)
            # Incrementar pos_index en uno
            pos_index = pos_index + 1

        # Segunda parte: Empaquetar n_to_take ejemplos negativos
        # Usando la misma lista del lote, comenzar desde neg_index e incrementar i hasta n_to_take
        for i in range(n_to_take):
            # Si el índice negativo sobrepasa la longitud del conjunto de datos negativo,
            if neg_index >= len_data_neg:
                # Si loop es False, romper una vez que llegamos al final del conjunto de datos
                if not loop:
                    stop = True;
                    break;
                # Si el usuario quiere seguir reutilizando los datos, reiniciar el índice
                neg_index = 0
                if shuffle:
                    # Mezclar el índice de la muestra negativa
                    rnd.shuffle(neg_index_lines)
            # obtener el tweet en pos_index
            tweet = data_neg[neg_index_lines[neg_index]]
            # convertir el tweet en tensores de enteros que representan las palabras procesadas
            tensor = tweet_to_tensor(tweet, vocab_dict)
            # agregar el tensor a la lista del lote
            batch.append(tensor)
            # Incrementar neg_index en uno
            neg_index += 1

        if stop:
            break;

        # Actualizar el índice de inicio para los datos positivos 
        # para que esté n_to_take posiciones después del pos_index actual
        pos_index += n_to_take
        
        # Actualizar el índice de inicio para los datos negativos 
        # para que esté n_to_take posiciones después del neg_index actual
        neg_index += n_to_take
        
        # Obtener la longitud máxima de tweet (la longitud del tweet más largo) 
        # (rellenarás todos los tweets más cortos para que tengan esta longitud)
        max_len = max([len(t) for t in batch]) 
        
        # Inicializar input_l, que 
        # almacenará las versiones rellenadas (padded) de los tensores
        tensor_pad_l = []
        # Rellenar los tweets más cortos con ceros
        for tensor in batch:

            # Obtener el número de posiciones a rellenar para este tensor para que tenga longitud max_len
            n_pad = max_len - len(tensor)
            
            # Generar una lista de ceros, con longitud n_pad
            pad_l = [0]*n_pad
            
            # concatenar el tensor y la lista de ceros de relleno
            tensor_pad = tensor + pad_l
            
            # agregar el tensor rellenado a la lista de tensores rellenados
            tensor_pad_l.append(tensor_pad)

        # convertir la lista de tensores rellenados a un arreglo de numpy
        # y almacenar esto como las entradas del modelo
        inputs = np.array(tensor_pad_l)
  
        # Generar la lista de objetivos para los ejemplos positivos (una lista de unos)
        # La longitud es el número de ejemplos positivos en el lote
        target_pos = [1]*n_to_take
        
        # Generar la lista de objetivos para los ejemplos negativos (una lista de unos)
        # La longitud es el número de ejemplos negativos en el lote
        target_neg = [0]*n_to_take
        
        # Concatenar los objetivos positivos y negativos
        target_l = target_pos + target_neg
        
        # Convertir la lista de objetivos en un arreglo de numpy
        targets = np.array(target_l)

        # Pesos de ejemplo: Tratar todos los ejemplos con igual importancia.
        example_weights = np.ones_like(targets)

        # nota: usamos yield y no return
        yield inputs, targets, example_weights

Ahora podemos usar el generador de datos para crear los datos de entrenamiento, y otro generador de datos para los datos de validación.

Crearemos un tercer generador de datos que no hace bucle (loop), para probar la exactitud final del modelo.

In [16]:
# Establecer el generador de números aleatorios para el procedimiento de mezcla (shuffle)
rnd.seed(30) 

# Crear el generador de datos de entrenamiento

# Crear el generador de datos de validación

# Crear el generador de datos de validación

# Obtener un lote del train_generator e inspeccionarlo.



Inputs: [[1995 4436 3188    9    0    0    0    0    0    0]
 [4938 1990 1443 5158 3486  141 3486  130  454    9]
 [3748  109  136  577 2917 3956    0    0    0    0]
 [ 247 3748    0    0    0    0    0    0    0    0]]
Targets: [1 1 0 0]
Example Weights: [1 1 1 1]


In [ ]:
# lista de 4 tensores rellenados con ceros


In [ ]:
# Probar el train_generator

# Crear un generador de datos para datos de entrenamiento,
# que produce lotes de tamaño 4 (para tensores y sus respectivos objetivos)


# Llamar al generador de datos para obtener un lote y sus objetivos


Ahora que tenemos los generadores de entrenamiento/validación, solo tenemos que llamarlos y devolverán tensores que corresponden a los tweets en la primera columna y sus etiquetas correspondientes en la segunda columna. Ahora podemos comenzar a construir la red neuronal. 

# Definir clases

En esta parte,crearemos nuestra propia librería de capas (layers). Será muy similar a la usada en Trax y también en Keras y PyTorch. Escribiremos un  pequeño framework que nos ayudará a entender cómo funcionan todos ellos y a usarlos eficazmente en el futuro.

El framework se basará en la siguiente clase `Layer` de utils.py.

```CPP
class Layer(object):
    """ Clase base para las capas.
    """
      
    # Constructor
    def __init__(self):
        # establecer los pesos en None
        self.weights = None

    # La propagación hacia adelante debe ser implementada
    # por las subclases de esta clase Layer
    def forward(self, x):
        raise NotImplementedError

    # Esta función inicializa los pesos
    # basándose en la firma de entrada y la clave aleatoria,
    # debe ser implementada por las subclases de esta clase Layer
    def init_weights_and_state(self, input_signature, random_key):
        pass

    # Esto inicializa y devuelve los pesos, no sobrescribir.
    def init(self, input_signature, random_key):
        self.init_weights_and_state(input_signature, random_key)
        return self.weights
 
    # __call__ permite que un objeto de esta clase
    # sea llamado como si fuera una función.
    def __call__(self, x):
        # Cuando se llama a este objeto de capa, 
        # llama a su función de propagación hacia adelante
        return self.forward(x)
```


## Clase ReLU
Implementaremos la función de activación ReLU en una clase a continuación. La función ReLU se ve de la siguiente manera: 
<img src = "Figures/relu.jpg" style="width:300px;height:150px;"/>

$$ \mathrm{ReLU}(x) = \mathrm{max}(0,x) $$

In [ ]:
class Relu(Layer):
    def forward(self, x):        
        activation = np.maximum(x,0)
        return activation

In [ ]:
# Probar la función relu
x = np.array([[-2.0, -1.0, 0.0], [0.0, 1.0, 2.0]], dtype=float)



<a name="3.2"></a>
## Clase Dense de la red

Implementaremos la función forward de la clase Dense. 
- La función forward multiplica la entrada de la capa (`x`) por la matriz de pesos (`W`)

$$\mathrm{forward}(\mathbf{x},\mathbf{W}) = \mathbf{xW} $$

- Podemos usar `numpy.dot` para realizar la multiplicación de matrices.

Hay que tener en cuenta que para una ejecución de código más eficiente, usaremos la versión de Trax de `math`, que incluye una versión de Trax de `numpy` y también `random`.

Implementaremos la función inicializadora de pesos `new_weights`
- Los pesos se inicializan con una clave aleatoria (random key).
- El segundo parámetro es una tupla para la forma deseada de los pesos (num_rows, num_cols)
- El número de filas de los pesos debe ser igual al número de columnas en x, porque para la propagación hacia adelante, multiplicarás x por los pesos.

Usaremos `trax.fastmath.random.normal(key, shape, dtype=tf.float32)` para generar valores aleatorios para la matriz de pesos. La diferencia clave entre esta función y la aleatoriedad estándar de `numpy` es el uso explícito de claves aleatorias, que deben pasarse. 

- `key` se puede generar llamando a `random.get_prng(seed=)` y pasando un número para la `seed`.
- `shape` es una tupla con la forma deseada de la matriz de pesos.
    - El número de filas en la matriz de pesos debe ser igual al número de columnas en la variable `x`. Dado que `x` puede tener 2 dimensiones si representa un solo ejemplo de entrenamiento (fila, columna), o tres dimensiones (batch_size, fila, columna), obtendremos la última dimensión de la tupla que contiene las dimensiones de x.
    - El número de columnas en la matriz de pesos es el número de unidades elegidas para esa capa densa. Mirar la función `__init__` para ver qué variable almacena el número de unidades.
- `dtype` es el tipo de dato de los valores en la matriz generada; mantén el valor por defecto de `tf.float32`. En este caso, no estableceremos explícitamente el dtype (solo dejaremos que se use el valor por defecto).

Estableceremos la desviación estándar de los valores aleatorios en 0.1
- Los valores generados tienen una media de 0 y una desviación estándar de 1.
- Estableceremos la desviación estándar `stdev` por defecto en 0.1 multiplicando la desviación estándar por cada uno de los valores en la matriz de pesos.

In [17]:
# usar el módulo fastmath dentro de trax
from trax import fastmath

# usar el módulo numpy de trax
np = fastmath.numpy

# usar el módulo fastmath.random de trax
random = fastmath.random

In [20]:
# Ver cómo funciona la función fastmath.trax.random.normal
tmp_key = random.get_prng(seed=1)
print("Dtos generados por random.get_prng")
display(tmp_key)

print("Escojemos una matriz de 2 filas con 3 columnas")
tmp_shape=(2,3)
display(tmp_shape)

# Generar una matriz de pesos
# Nota: obtendrás un error si intentas establecer dtype en tf.float32, donde tf es tensorflow
# Simplemente evita establecer el dtype y permite que use el tipo de dato por defecto
tmp_weight = trax.fastmath.random.normal(key=tmp_key, shape=tmp_shape)

print("Matriz de pesos generada con una distribución normal con media 0 y desviación estándar de 1.")
display(tmp_weight)

Dtos generados por random.get_prng


Array([0, 1], dtype=uint32)

Escojemos una matriz de 2 filas con 3 columnas


(2, 3)

Matriz de pesos generada con una distribución normal con media 0 y desviación estándar de 1.


Array([[-0.15443718,  0.08470728, -0.13598049],
       [-0.15503626,  1.2666674 ,  0.14829758]], dtype=float32)

**Implementemos la clase `Dense`.**

In [21]:
class Dense(Layer):
    
    #Una capa densa (totalmente conectada).
    def __init__(self, n_units, init_stdev=0.1):
        # Establecer el número de unidades en esta capa
        self._n_units = n_units
        self._init_stdev = init_stdev

    def forward(self, x):
        # Multiplicar matricialmente x y la matriz de pesos
        dense = np.dot(x, self.weights) 
        return dense

    # init_weights
    def init_weights_and_state(self, input_signature, random_key):
        # input_signature tiene un atributo .shape que da la forma como una tupla
        input_shape = input_signature.shape

        # Generar la matriz de pesos a partir de una distribución normal, 
        # y desviación estándar de 'stdev'        
        w = self._init_stdev * random.normal(key = random_key, shape = (input_shape[-1], self._n_units))
        
        self.weights = w
        return self.weights

In [ ]:
# Probar la capa Dense
z = np.array([[2.0, 7.0, 25.0]]) # arreglo de entrada 

#establecer el número de unidades en la capa densa

#establecer la semilla aleatoria

#Inicializzar los pesos



In [ ]:
#Devuelve los pesos generados aleatoriamente


In [ ]:
# Devuelve los valores multiplicados de unidades y pesos

<a name="3.3"></a>
##  Modelo Neuronal 

Ahora implementaremos un clasificador usando redes neuronales. Aquí está la arquitectura del modelo:. 

<img src = "Figures/nn.jpg" style="width:400px;height:250px;"/>

Para la implementación del modelo, usaremos la librería de capas de Trax `tl`.
Hay que tener en cuenta que el segundo carácter de `tl` es la letra `L` minúscula, no el número 1. Las capas de Trax son muy similares a las que implementamod arriba, pero además de los pesos entrenables también tienen un estado no entrenable.

El estado se usa en capas como la normalización por lotes (batch normalization) y para la inferencia


In [ ]:
# Ver la documentación de tl.Dense
#help(tl.Dense)

In [ ]:
# Ver la documentación de tl.Serial
#help(tl.Serial)

- [tl.Embedding](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L113): Función constructora de capa para una capa de embedding.  
    - `tl.Embedding(vocab_size, d_feature)`.
    - `vocab_size` es el número de palabras únicas en el vocabulario dado.
    - `d_feature` es el número de elementos en el embedding de palabra (algunas opciones para el tamaño de un embedding de palabra van de 150 a 300, por ejemplo).

In [ ]:
# Ver la documentación de tl.Embedding
#help(tl.Embedding)

In [ ]:
#ejemplo de una capa embedding


- [tl.Mean](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L276): Calcula medias a lo largo de un eje. En este caso, elegiremos axis = 1 para obtener un vector de embedding promedio (un vector de embedding que es un promedio de todas las palabras del vocabulario).  
- Por ejemplo, si la matriz de embedding tiene 300 elementos y el tamaño del vocabulario es de 10,000 palabras, tomar la media de la matriz de embedding a lo largo de axis=1 producirá un vector de 300 elementos.

In [ ]:
# ver la documentación de tl.mean
#help(tl.Mean)

In [22]:
# Imagina que la matriz de embedding usa 
# 2 elementos para representar el significado de una palabra
# y tiene un tamaño de vocabulario de 3
# Así que tiene forma (2,3)
tmp_embed = np.array([[1,2,3,],
                    [4,5,6]
                   ])

# tomar la media a lo largo del axis 0
print("La media a lo largo del eje 0 genera un vector cuya longitud es igual al tamaño del vocabulario.")
display(np.mean(tmp_embed,axis=0))

print("La media a lo largo del eje 1 genera un vector cuya longitud es igual al número de elementos de un word embedding.")
display(np.mean(tmp_embed,axis=1))

La media a lo largo del eje 0 genera un vector cuya longitud es igual al tamaño del vocabulario.


Array([2.5, 3.5, 4.5], dtype=float32)

La media a lo largo del eje 1 genera un vector cuya longitud es igual al número de elementos de un word embedding.


Array([2., 5.], dtype=float32)

- [tl.LogSoftmax](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L242): Implementemos la función log softmax
- Aquí, no necesitamos establecer ningún parámetro para `LogSoftMax()`.

In [ ]:
#help(tl.LogSoftmax)

**Documentación en línea**

- [tl.Dense](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Dense)

- [tl.Serial](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#module-trax.layers.combinators)

- [tl.Embedding](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Embedding)

- [tl.Mean](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Mean)

- [tl.LogSoftmax](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.LogSoftmax)

**Implementando la función del clasificador.**

In [ ]:
def classifier(vocab_size=len(Vocab), embedding_dim=256, output_dim=2, mode='train'):
    # crear la capa de embedding
    
    # Crear una capa de media (mean), para crear un embedding de palabra "promedio"
    
    # Crear una capa densa, una unidad por cada salida
    
    # Crear la capa log softmax (no se necesitan parámetros)
    
    # Usar tl.Serial para combinar todas las capas
    # y crear el clasificador
    # de tipo trax.layers.combinators.Serial
    
    # devolver el modelo del tipo
    return model

In [ ]:
# Porbar la función


# Entrenamiento

Para entrenar un modelo en una tarea, Trax define una abstracción [`trax.supervised.training.TrainTask`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.TrainTask) que empaqueta los datos de entrenamiento, la pérdida y el optimizador (entre otras cosas) juntos en un objeto.

De manera similar, para evaluar un modelo, Trax define una abstracción [`trax.supervised.training.EvalTask`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.EvalTask) que empaqueta los datos de evaluación y las métricas (entre otras cosas) en otro objeto.

La pieza final que une todo es la abstracción [`trax.supervised.training.Loop`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.Loop) que es una forma muy simple y flexible de juntar todo y entrenar el modelo, todo mientras lo evalúa y guarda puntos de control (checkpoints).
Usar `Loop` nos ahorrará mucho código comparado con escribir siempre el bucle de entrenamiento a mano. Más importante aún, es menos probable que tengamos un error en ese código que arruine el entrenamiento.

In [ ]:
# Ver la documentación de trax.supervised.training.TrainTask
#help(trax.supervised.training.TrainTask)

In [ ]:
# Ver la documentación de trax.supervised.training.EvalTask
#help(trax.supervised.training.EvalTask)

In [ ]:
# Ver la documentación de trax.supervised.training.Loop
#help(trax.supervised.training.Loop)

In [ ]:
# Ver los optimizadores entre los que podrías elegir
#help(trax.optimizers)

Observa que algunos optimizadores disponibles incluyen:
```CPP
    adafactor
    adam
    momentum
    rms_prop
    sm3
```

<a name="4.1"></a>
## Entrenar el modelo

Ahora vas a entrenar nuestro modelo. 

Definamos `TrainTask`, `EvalTask` y `Loop` como preparación para entrenar el modelo.

In [ ]:
from trax.supervised import training

batch_size = 16
rnd.seed(271)

train_task = training.TrainTask(
    labeled_data=train_generator(batch_size=batch_size, shuffle=True),
    loss_layer=tl.CrossEntropyLoss(),
    optimizer=trax.optimizers.Adam(0.01),
    n_steps_per_checkpoint=10,
)

eval_task = training.EvalTask(
    labeled_data=val_generator(batch_size=batch_size, shuffle=True),
    metrics=[tl.CrossEntropyLoss(), tl.Accuracy()],
)

model = classifier()

Esto define un modelo entrenado usando [`tl.CrossEntropyLoss`](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.CrossEntropyLoss) optimizado con el optimizador [`trax.optimizers.Adam`](https://trax-ml.readthedocs.io/en/latest/trax.optimizers.html#trax.optimizers.adam.Adam), todo mientras se rastrea la exactitud usando la métrica [`tl.Accuracy`](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.Accuracy). También rastreamos `tl.CrossEntropyLoss` en el conjunto de validación.

Ahora hagamos un directorio de salida y entrenemos el modelo.

In [ ]:
output_dir = '~/model/'
output_dir_expand = os.path.expanduser(output_dir)
print(output_dir_expand)

Implementaremos `train_model` para entrenar el modelo (`classifier` que implementamos antes) durante el número dado de pasos de entrenamiento (`n_steps`) usando `TrainTask`, `EvalTask` y `Loop`.

In [ ]:
def train_model(classifier, train_task, eval_task, n_steps, output_dir):   
    training_loop = training.Loop(
                                classifier, # The learning model
                                train_task, # The training task
                                eval_tasks = [eval_task], # The evaluation task
                                output_dir = output_dir) # The output directory

    training_loop.run(n_steps = n_steps)

    return training_loop

In [ ]:
#Entrenar el modelo con train_model


## Hacer una predicción

Ahora que hemos entrenado un modelo, podemos acceder a él como el objeto `training_loop.model`. Por ahora, haremos predicciones con nuestro modelo.

Usamos los datos de entrenamiento solo para ver cómo funciona el proceso de predicción.  
- Más adelante, usaremos datos de validación para evaluar el rendimiento de tu modelo.

In [ ]:
# Crear un objeto generador
tmp_train_generator = train_generator(16)

# obtener un lote
tmp_batch = next(tmp_train_generator)

# La posición 0 tiene las entradas del modelo (tweets como tensores)
# la posición 1 tiene los objetivos (las etiquetas reales)
tmp_inputs, tmp_targets, tmp_example_weights = tmp_batch


In [ ]:
# alimentar los tensores de tweets al modelo para obtener una predicción


Para convertir estas probabilidades en categorías (predicción de sentimiento negativo o positivo), para cada fila:
- Comparamos las probabilidades en cada columna.
- Si la columna 1 tiene un valor mayor que la columna 0, clasifica eso como un tweet positivo.
- De lo contrario, si la columna 1 es menor o igual que la columna 0, clasifica ese ejemplo como un tweet negativo.

In [ ]:
# convertir probabilidades en predicciones de categoría
tmp_is_positive = tmp_pred[:,1] > tmp_pred[:,0]
for i, p in enumerate(tmp_is_positive):
    print(f"Neg log prob {tmp_pred[i,0]:.4f}\tPos log prob {tmp_pred[i,1]:.4f}\t is positive? {p}\t actual {tmp_targets[i]}")

Observemos que dado que estamos haciendo una predicción usando un lote de entrenamiento, es más probable que las predicciones del modelo coincidan con los objetivos reales (etiquetas).  
- Cada predicción de que el tweet es positivo también coincide con el objetivo real de 1 (sentimiento positivo).
- De manera similar, todas las predicciones de que el sentimiento no es positivo coinciden con el objetivo real de 0 (sentimiento negativo).

Otra cosa útil que conviene saber es cómo comparar si la predicción coincide con el objetivo real (etiqueta).  
- El resultado del cálculo `is_positive` es un booleano.
- El objetivo es de tipo trax.fastmath.numpy.int32
- Si esperamos hacer divisiones, quizás prefieras trabajar con números decimales con el tipo de dato trax.fastmath.numpy.int32

In [ ]:
# Ver el arreglo de booleanos
print("Arreglo de booleans")
display(tmp_is_positive)

# convertir booleano a tipo int32
# True se convierte en 1
# False se convierte en 0
tmp_is_positive_int = tmp_is_positive.astype(np.int32)


# Ver el arreglo de enteros
print("Arreglo de integers")
display(tmp_is_positive_int)

# convertir booleano a tipo float32
tmp_is_positive_float = tmp_is_positive.astype(np.float32)

# Ver el arreglo de flotantes
print("Arreglo de floats")
display(tmp_is_positive_float)

Hay que tener en cuenta que Python normalmente hace la conversión de tipos por ti cuando comparas un booleano con un entero
- True comparado con 1 es True, de lo contrario cualquier otro entero es False.
- False comparado con 0 es True, de lo contrario cualquier otro entero es False.

In [ ]:
print(f"True == 1: {True == 1}")
print(f"True == 2: {True == 2}")
print(f"False == 0: {False == 0}")
print(f"False == 2: {False == 2}")

Sin embargo, hay que llevar un registro del tipo de dato de las variables para evitar resultados inesperados. Por eso ayuda convertir los booleanos en enteros
- Compararemos 1 con 1 en lugar de comparar True con 1.

# Evaluación  
## Calcular la exactitud en un lote

Ahora escribiremos una función que evalúa el modelo en el conjunto de validación y devuelve la exactitud. 
- `preds` contiene las predicciones.
    - Sus dimensiones son `(batch_size, output_dim)`. `output_dim` es dos en este caso. La columna 0 contiene la probabilidad de que el tweet pertenezca a la clase 0 (sentimiento negativo). La columna 1 contiene la probabilidad de que pertenezca a la clase 1 (sentimiento positivo).
    - Si la probabilidad en la columna 1 es mayor que la probabilidad en la columna 0, interpreta esto como la predicción del modelo de que el ejemplo tiene la etiqueta 1 (sentimiento positivo).  
    - De lo contrario, si las probabilidades son iguales o la probabilidad en la columna 0 es mayor, la predicción del modelo es 0 (sentimiento negativo).
- `y` contiene las etiquetas reales.
- `y_weights` contiene los pesos a dar a las predicciones.

In [ ]:
def compute_accuracy(preds, y, y_weights):
    # Crear un arreglo de booleanos, 
    # True si la probabilidad de sentimiento positivo es mayor que
    # la probabilidad de sentimiento negativo
    # de lo contrario False
    is_pos =  preds[:, 1] > preds[:, 0] 

    # convertir el arreglo de booleanos en un arreglo de np.int32
    is_pos_int = is_pos.astype(np.int32)
    
    # comparar el arreglo de predicciones (como int32) con el objetivo (etiquetas) de tipo int32
    correct = is_pos_int == y

    # Contar la suma de los pesos.
    sum_weights = np.sum(y_weights)
    
    # convertir el arreglo de predicciones correctas (booleano) en un arreglo de np.float32
    correct_float = correct.astype(np.float32)
    
    # Multiplicar cada predicción por su peso correspondiente.
    weighted_correct_float = correct_float * y_weights

    # Sumar las predicciones correctas ponderadas (de tipo np.float32), para ir en el
    # denominador.
    weighted_num_correct = np.sum(weighted_correct_float)
 
    # Dividir el número de predicciones correctas ponderadas por la suma de los
    # pesos.
    accuracy = weighted_num_correct / sum_weights

    return accuracy, weighted_num_correct, sum_weights

In [ ]:
# probar la función
tmp_val_generator = val_generator(64)

# obtener un lote
tmp_batch = next(tmp_val_generator)

# La posición 0 tiene las entradas del modelo (tweets como tensores)
# la posición 1 tiene los objetivos (las etiquetas reales)
tmp_inputs, tmp_targets, tmp_example_weights = tmp_batch

# alimentar los tensores de tweets al modelo para obtener una predicción
tmp_pred = training_loop.eval_model(tmp_inputs)

tmp_acc, tmp_num_correct, tmp_num_predictions = compute_accuracy(preds=tmp_pred, y=tmp_targets, y_weights=tmp_example_weights)

print(f"La precisión de la predicción del modelo en un único lote de entrenamiento es: {100 * tmp_acc}%")
print(f"Número ponderado de predicciones correctas {tmp_num_correct}; número ponderado del total de observaciones predecidas {tmp_num_predictions}")

## Probar el modelo con datos de validación

Ahora escribiremos una prueba de la exactitud de predicción de nuestro modelo con datos de validación. 

Esta función recibirá un generador de datos y el modelo. 
- El generador permite obtener lotes de datos. Puedes usarlo con un bucle `for`:

```
for batch in iterator: 
   # hacer algo con ese lote
```

`batch` tiene dimensiones `(X, Y, weights)`. 
- La columna 0 corresponde al tweet como tensor (entrada).
- La columna 1 corresponde a su objetivo (etiqueta real, sentimiento positivo o negativo).
- La columna 2 corresponde a los pesos asociados (pesos de ejemplo)
- Puedes alimentar el tweet al modelo y devolverá las predicciones para el lote. 

- Calcularemos la exactitud sobre todos los lotes en el iterador de validación. 
- Haremos uso de `compute_accuracy`, que implementaste recientemente, y devuelve la exactitud general.

In [ ]:
def test_model(generator, model):
    
    accuracy = 0.
    total_num_correct = 0
    total_num_pred = 0
    
    for batch in generator: 
        
        # Recuperar las entradas del lote
        inputs = batch[0]
        
        # Recuperar los objetivos (etiquetas reales) del lote
        targets = batch[1]
        
        # Recuperar el peso del ejemplo.
        example_weight = batch[2]

        # Hacer predicciones usando las entradas
        pred = model(inputs)
        
        # Calcular la exactitud para el lote comparando sus predicciones y objetivos
        batch_accuracy, batch_num_correct, batch_num_pred = compute_accuracy(pred, targets, example_weight)
        
        # Actualizar el número total de predicciones correctas
        # sumando el número de predicciones correctas de este lote
        total_num_correct += batch_num_correct
        
        # Actualizar el número total de predicciones 
        # sumando el número de predicciones hechas para el lote
        total_num_pred += batch_num_pred

    # Calcular la exactitud sobre todos los ejemplos
    accuracy = total_num_correct / total_num_pred
    return accuracy

In [ ]:
# probando la exactitud de tu modelo: esto toma alrededor de 20 segundos
model = training_loop.eval_model
accuracy = test_model(test_generator(16), model)

print(f'La precisión del modelo en el conjunto de validación es {accuracy:.4f}', )

# Probar el modelo con un tweet nuevo

Finalmente probaremos el modelo con un nuevo tweet. Veremos que las redes profundas son más potentes que los modelos anteriores que creamos (regresión logística, Naive Bayes).

In [ ]:
# esto se usa para predecir sobre tu propia oración
def predict(sentence):
    inputs = np.array(tweet_to_tensor(sentence, vocab_dict=Vocab))
    
    # Tamaño de lote 1, agregar dimensión para el lote, para trabajar con el modelo
    inputs = inputs[None, :]  
    
    # predecir con el modelo
    preds_probs = model(inputs)
    
    # Convertir probabilidades en categorías
    preds = int(preds_probs[0, 1] > preds_probs[0, 0])
    
    sentiment = "negative"
    if preds == 1:
        sentiment = 'positive'

    return preds, sentiment


In [ ]:
# probar una oración positiva
sentence = "It's such a nice day, think i'll be taking Sid to Ramsgate fish and chips for lunch at Peter's fish factory and then the beach maybe"


In [ ]:

print()
# probar una oración negativa
sentence = "I hated my day, it was the worst, I'm so sad."


Observa que el modelo funciona bien incluso para oraciones complejas.

### Sobre las Redes Profundas (Deep Nets)

Las redes profundas nos permiten entender y capturar dependencias que no habrías podido capturar con una simple regresión lineal, o regresión logística. 
- También te permiten usar mejor los embeddings preentrenados para clasificación y tienden a generalizar mejor.